In [1]:
import os

import requests
import pandas as pd
import time

def fetch_data(topic, output_filename="scaped.csv", limit = 10000):
    headers = {
        'User-Agent': 'shauryachauhan2050@gmail.com'
    }

    base_url = "https://api.openalex.org/works"
    
    params = {
        'search': topic,
        'filter': "has_abstract:true",
        'per_page':200,
        'cursor': '*',
        'api_key': os.environ.get('OPENALEX_API_KEY')
    }

    columns = [
        'openalex_id', 'title', 'doi', 'abstract', 'publish_time',
        'authors', 'journal', 'pmcid', 'pubmed_id'
    ]

    pd.DataFrame(columns = columns).to_csv(output_filename, index=False)
    print(f"Initialized clean file: {output_filename}")
    total_collected = 0

    while params['cursor'] and total_collected < limit:
        try:
            print(f"Progress: Fetching next batch... Total saved so far: {total_collected} / {limit}")
            response = requests.get(base_url, params = params, headers = headers)
            response.raise_for_status()
            data = response.json()
            
            results = data.get('results', [])
            if not results:
                print("no more results available.")
                break
            
            batch_data = []

            for paper in results:
                abstract_index = paper.get('abstract_inverted_index', {})
                abstract = ""
                if abstract_index:
                    words = []
                    for word, positions in abstract_index.items():
                        for pos in positions:
                            words.append((pos, word))
                    words.sort()
                    abstract = ' '.join([word for _, word in words])
            
                if not abstract.strip():
                    continue
                
                authors = [auth.get('author', {}).get('display_name', '')
                           for auth in paper.get('authorships', [])]
                authors = [name for name in authors if name]

                journal = ''
                primary_location = paper.get('primary_location')
                if primary_location and primary_location.get('source'):
                    journal = primary_location['source'].get('display_name', '')

                batch_data.append({
                    'openalex_id': paper.get('id', ''),
                    'title': paper.get('title', ''),
                    'doi': paper.get('doi', ''),
                    'abstract': abstract,
                    'publish_time': paper.get('publication_date', ''),
                    'authors': '; '.join(authors),
                        'journal': journal,
                    'pmcid': paper.get('ids', {}).get('pmcid', ''),
                    'pubmed_id': paper.get('ids', {}).get('pmid', '')
                })

                total_collected += 1
                if total_collected >= limit:
                    break
        
            if batch_data:
                batch_df = pd.DataFrame(batch_data, columns = columns)
                batch_df.to_csv(output_filename, mode='a', header=False, index=False)
        
            params['cursor'] = data.get('meta', {}).get('next_cursor')
            time.sleep(0.1)

        except Exception as e:
            print(f"\nAn error occurred: {e}")
            break
    
    print(f"\nSuccess! Process finished. {total_collected} rows saved in total.")


In [2]:
if __name__ == "__main__":
    fetch_data(
        topic = "Space Biology",
        output_filename = "scraped.csv",
        limit = 400
    )

Initialized clean file: scraped.csv
Progress: Fetching next batch... Total saved so far: 0 / 400
Progress: Fetching next batch... Total saved so far: 200 / 400

Success! Process finished. 400 rows saved in total.


In [17]:
import pandas as pd
import numpy as np
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

In [18]:
data = pd.read_csv("scraped.csv", encoding='latin-1')
data.head(3)

,openalex_id,title,doi,abstract,publish_time,authors,journal,pmcid,pubmed_id
0,https://openalex.org/W2110829397,PROBLEMS OF SPACE BIOLOGY,NaN,Space biology - effect of radiation exposure a...,1963-03-25,N. M. Sisakyan,Defense Technical Information Center (DTIC),NaN,NaN
1,https://openalex.org/W2035753075,Chemical space and biology,https://doi.org/10.1038/nature03192,Chemical space--which encompasses all possible...,2004-12-15,Christopher M. Dobson,Nature,NaN,https://pubmed.ncbi.nlm.nih.gov/15602547
2,https://openalex.org/W2988387100,Space biology and medicine.,NaN,Perhaps one of the greatest gifts that has bee...,1979-10-01,Arnauld Nicogossian; Stanley R. Mohler; Ð. Ð...,PubMed,NaN,https://pubmed.ncbi.nlm.nih.gov/11902165


In [19]:
df_csv = data
df_csv = df_csv[['openalex_id','title','doi','abstract','publish_time','authors','journal','doi','pmcid','pubmed_id']]
df_csv.head()

,openalex_id,title,doi,abstract,publish_time,authors,journal,doi,pmcid,pubmed_id
0,https://openalex.org/W2110829397,PROBLEMS OF SPACE BIOLOGY,NaN,Space biology - effect of radiation exposure a...,1963-03-25,N. M. Sisakyan,Defense Technical Information Center (DTIC),NaN,NaN,NaN
1,https://openalex.org/W2035753075,Chemical space and biology,https://doi.org/10.1038/nature03192,Chemical space--which encompasses all possible...,2004-12-15,Christopher M. Dobson,Nature,https://doi.org/10.1038/nature03192,NaN,https://pubmed.ncbi.nlm.nih.gov/15602547
2,https://openalex.org/W2988387100,Space biology and medicine.,NaN,Perhaps one of the greatest gifts that has bee...,1979-10-01,Arnauld Nicogossian; Stanley R. Mohler; Ð. Ð...,PubMed,NaN,NaN,https://pubmed.ncbi.nlm.nih.gov/11902165
3,https://openalex.org/W3114753755,Advantages and Limitations of Current Microgra...,https://doi.org/10.3390/app11010068,Human Space exploration has created new challe...,2020-12-23,Francesca Ferranti; Marta Del Bianco; Claudia ...,Applied Sciences,https://doi.org/10.3390/app11010068,NaN,NaN
4,https://openalex.org/W371744478,AN UPDATE ON PLANT SPACE BIOLOGY,NaN,Space craft in low-Earth orbit can provide a m...,2011-08-30,Chris Wolverton; John Z. Kiss,Digital Commons - OWU (Ohio Wesleyan University),NaN,NaN,NaN


In [20]:
print(len(df_csv))
df_csv = df_csv[~df_csv['abstract'].isnull()]
print(len(df_csv))

199993
199989


In [21]:
df_csv['abstract'] = df_csv['abstract'].apply(lambda x:
                                          x.replace('BACKGROUND:','').replace('BACKGROUNDS:','').replace('OBJECTIVES:','')
                                          .replace('OBJECTIVE:','').replace('METHODS:','').replace('METHOD:','')
                                          .replace('RESULTS:','').replace('RESULT:','')
                                          .replace('CONCLUSION:','').replace('CONCLUSIONS:',''))

In [22]:
df_csv['abstract'] = df_csv['abstract'].apply(lambda x: x.lower())
df_csv['abstract'] = df_csv['abstract'].apply(lambda x: x.replace('this article is protected by copyright. all rights reserved',''))

In [23]:
df_csv.to_csv('abstract_final.csv', index=None)

In [24]:
print(len(df_csv))
df_csv = df_csv[~df_csv['publish_time'].isnull()]
print(len(df_csv))

199989
199921


In [25]:
df = df_csv
df['publish_time_new'] = pd.to_datetime(df['publish_time'], format='%Y-%m-%d',errors='coerce')

In [26]:
print(len(df))
df = df[df.publish_time_new != "NaT"]
print(len(df))

199921
199921


In [11]:
#OPTIONAL
import datetime
df= df[df['publish_time_new']>'2020-01-01']
len(df)

48373

In [27]:
# IF NOT USED: USE LANGUAGE = MULTILINGUAL IN TRAINING THE MODEL
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0
def langdet (x):
    try:
        return detect(x)
    except:
        return "NA"
df['lang'] = df['abstract'].apply(lambda x: langdet(x))
df = df[df['lang'].str.contains('en')]
df.to_csv('articles_clean_text_eng.csv', index=None)
len(df)

199102

In [28]:
import pandas as pd
df = pd.read_csv('articles_clean_text_eng.csv')

In [29]:
df1 = df
len(df1) # MUST BE SAME AS THE PREVIOUS STEP

199102

In [30]:
unk = """'Unknown'
'Not Known'
'Little is known'
'Unrevealed'
'Uncertain'
'Undetermined'
'Understudied'
'Unexplored'
'Unclear'
'Not fully understood'
'Literature gap'
'Research gap'
'Knowledge gap'
'Future studies'
'Future research'
'Research problem'
'More studies'
'More research'
'Further studies'
'Further research'"""
unk = unk.replace("\n",",").replace("'","").lower().split(",")
unk= "|".join(unk)
unk

'unknown|not known|little is known|unrevealed|uncertain|undetermined|understudied|unexplored|unclear|not fully understood|literature gap|research gap|knowledge gap|future studies|future research|research problem|more studies|more research|further studies|further research'

In [31]:
df2 = df1[df1.abstract.str.contains(unk)]
len(df2)

25535

In [32]:
# Commented lines allow for local stopword database.
import re
import nltk
import string
from textblob import TextBlob
# nltk.download('stopwords')
# nltk.download('wordnet')
stopword = nltk.corpus.stopwords.words('english')
#my_file = open("stopwords.txt", "r")
#content = my_file.read().split('\n')
#stopword.extend(content)
stopword = list(set(stopword))
stopword = [w.strip() for w in stopword]
stopword = set(stopword)
wn = nltk.WordNetLemmatizer()
ps = nltk.PorterStemmer()
from nltk import bigrams, trigrams

def removeWeirdChars(text):
    weirdPatterns = re.compile("["u"\U0001F600-\U0001F64F"u"\U0001F300-\U0001F5FF"u"\U0001F680-\U0001F6FF"u"\U0001F1E0-\U0001F1FF"u"\U00002702-\U000027B0"u"\U000024C2-\U0001F251"u"\U0001f926-\U0001f937"u'\U00010000-\U0010ffff'
                               u"\u200d"u"\u2640-\u2642"u"\u2600-\u2B55"u"\u23cf"u"\u23e9"u"\u231a"u"\u3030"u"\ufe0f"u"\u2069"u"\u2066"u"\u200c"u"\u2068"u"\u2067""]+", flags=re.UNICODE)
    return weirdPatterns.sub(r'', text)
def remove_punct(text):
    text  = "".join([char for char in text if char not in string.punctuation])
    text = re.sub('[0-9]+', '', text)
    return text
def tokenization(text):
    text = text.split()#re.split('\W+', text)
    text = ','.join(set(text))
    return text
def remove_stopwords(text):
    text = [word.strip() for word in text.split() if word not in stopword]
    text = ' '.join(text)
    return text
def stemming(text): # This is unused
    text = [ps.stem(word) for word in text.split()]
    text = ' '.join(text)
    return text

def lemmatizer(text): # This is unused
    text = [wn.lemmatize(word) for word in text.split()]
    text = ' '.join(text)
    return text
def clean_text(text):
    text_lc = " ".join([word.lower() for word in text.split() if word not in string.punctuation]) # remove puntuation
    text_rc = re.sub('[0-9]+', '', text_lc)
    tokens = re.split('\W+', text_rc)    # tokenization
    text = [word for word in tokens if word not in stopword]  # remove stopwords and stemming
    text = ' '.join(text)
    return text


df2['clean_text'] = df2['abstract'].apply(lambda x: clean_text(x))

In [ ]:
from nltk.tokenize import sent_tokenize
nltk.download('punkt_tab')

wl = unk.split("|")
def sentence(x):
    s, w = [], []
    sent = sent_tokenize(x)
    if len(sent)<=1:
        s, w =["-1"], ["-1"]
    else:
        for c, i in enumerate(sent):
            for j in wl:
                if j in i:
                    w.append(j)
                    if c==0 or c==len(sent)-1:
                        s.append(i)
                    else:
                        s.append(sent[c-1])
                        s.append(sent[c])
                        s.append(sent[c+1])
    s = list(dict.fromkeys(s))
    w = list(dict.fromkeys(w))
    C = ".".join(s)
    D = ",".join(w)
    return pd.Series([C, D])

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\shaur\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [34]:
df2[['sentences', 'words']] = df2['abstract'].apply(sentence)

In [37]:
df2.head()

,openalex_id,title,doi,abstract,publish_time,authors,journal,doi.1,pmcid,pubmed_id,publish_time_new,lang,clean_text,sentences,words
6,https://openalex.org/W2160719116,Synthetic biology for the directed evolution o...,https://doi.org/10.1039/c4cs00351a,the amino acid sequence of a protein affects b...,2014-12-15,Andrew Currin; Neil Swainston; Philip J. Day; ...,Chemical Society Reviews,https://doi.org/10.1039/c4cs00351a,NaN,https://pubmed.ncbi.nlm.nih.gov/25503938,2014-12-15,en,amino acid sequence protein affects structure ...,enzymologists distinguish binding (kd) and cat...,unknown
8,https://openalex.org/W3016143870,Space Radiation Biology for âLiving in Spaceâ,https://doi.org/10.1155/2020/4703286,space travel has advanced significantly over t...,2020-01-01,Satoshi Furukawa; Aiko Nagamatsu; Mitsuru Neno...,BioMed Research International,https://doi.org/10.1155/2020/4703286,NaN,https://pubmed.ncbi.nlm.nih.gov/32337251,2020-01-01,en,space travel advanced significantly last six d...,future research that furthers our understandin...,future research
26,https://openalex.org/W1424654272,Planning Algorithms,https://doi.org/10.1017/cbo9780511546877,planning algorithms are impacting technical di...,2006-05-29,Steven M. LaValle,Cambridge University Press eBooks,https://doi.org/10.1017/cbo9780511546877,NaN,NaN,2006-05-29,en,planning algorithms impacting technical discip...,the treatment is centered on robot motion plan...,uncertain
28,https://openalex.org/W4366735078,Perspectives for plant biology in space and an...,https://doi.org/10.1038/s41526-023-00315-x,advancements in plant space biology are requir...,2023-08-21,Veronica De Micco; Giovanna Aronne; Nicol Capl...,npj Microgravity,https://doi.org/10.1038/s41526-023-00315-x,NaN,https://pubmed.ncbi.nlm.nih.gov/37604914,2023-08-21,en,advancements plant space biology required real...,this apparent paradox is a current research ch...,knowledge gap
58,https://openalex.org/W2056901756,Plant biology in space: recent accomplishments...,https://doi.org/10.1111/plb.12127,gravity has shaped the evolution of life since...,2013-12-23,GÃ¼nter Ruyters; Markus Braun,Plant Biology,https://doi.org/10.1111/plb.12127,NaN,https://pubmed.ncbi.nlm.nih.gov/24373009,2013-12-23,en,gravity shaped evolution life since origin how...,a chapter summarising major scientific breakth...,future research


In [38]:
len(df2),len(df2.pubmed_id==None)

(25535, 25535)

In [39]:
df2.to_csv("articles_with_sentences and words1.csv", index=None)

# Data


In [1]:
import pandas as pd
df2 = pd.read_csv('articles_with_sentences and words1.csv')

In [2]:
df2.head(2)

,openalex_id,title,doi,abstract,publish_time,authors,journal,doi.1,pmcid,pubmed_id,publish_time_new,lang,clean_text,sentences,words
0,https://openalex.org/W2160719116,Synthetic biology for the directed evolution o...,https://doi.org/10.1039/c4cs00351a,the amino acid sequence of a protein affects b...,2014-12-15,Andrew Currin; Neil Swainston; Philip J. Day; ...,Chemical Society Reviews,https://doi.org/10.1039/c4cs00351a,NaN,https://pubmed.ncbi.nlm.nih.gov/25503938,2014-12-15,en,amino acid sequence protein affects structure ...,enzymologists distinguish binding (kd) and cat...,unknown
1,https://openalex.org/W3016143870,Space Radiation Biology for âLiving in Spaceâ,https://doi.org/10.1155/2020/4703286,space travel has advanced significantly over t...,2020-01-01,Satoshi Furukawa; Aiko Nagamatsu; Mitsuru Neno...,BioMed Research International,https://doi.org/10.1155/2020/4703286,NaN,https://pubmed.ncbi.nlm.nih.gov/32337251,2020-01-01,en,space travel advanced significantly last six d...,future research that furthers our understandin...,future research


In [3]:
print(len(df2))


# Drop duplicates only on non-null values

print(len(df2))
df2 = df2.drop_duplicates(subset=['openalex_id'], keep='first')
print(f"After deduplicating on openalex_id: {len(df2)}")

cols_to_dedup = ['doi', 'pmcid', 'pubmed_id']

for col in cols_to_dedup:
    df_with_val = df2[df2[col].notna()].drop_duplicates(subset=[col], keep='first')
    df_without_val = df2[df2[col].isna()]
    df2 = pd.concat([df_with_val, df_without_val]).reset_index(drop=True)
    print(f"After deduplicating on {col}: {len(df2)}")

25535
25535
After deduplicating on openalex_id: 25535
After deduplicating on doi: 25520
After deduplicating on pmcid: 25520
After deduplicating on pubmed_id: 25520


In [4]:
#docs = df2[df2['clean_text'].notna()].clean_text.to_list() 
# OR IF OVER 70,000 or so are left: 
docs = df2[df2['sentences'].notna()].clean_text.to_list()

In [5]:
docs[2]

print(len(docs))

25520


# **Topic Modeling**




In [ ]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired

representation_model = KeyBERTInspired()

# Inject it into your BERTopic initialization
topic_model = BERTopic(
    language="english", # OR language="multilingual"
    calculate_probabilities=True, 
    verbose=True
    representation_model = representation_model
)

topics, probs = topic_model.fit_transform(docs)

c:\Users\shaur\OneDrive\Desktop\Research Gap Identification\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-03 23:50:42,203 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 798/798 [03:50<00:00,  3.46it/s]
2026-06-03 23:54:36,120 - BERTopic - Embedding - Completed ✓
2026-06-03 23:54:36,120 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-03 23:54:50,306 - BERTopic - Dimensionality - Completed ✓
2026-06-03 23:54:50,308 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-03 23:55:38,727 - BERTopic - Cluster - Completed ✓
2026-06-03 23:55:38,732 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-03 23:55:40,372 - BERTopic - Representation - Completed ✓


## Extracting Topics


In [7]:
freq = topic_model.get_topic_info(); freq.head(15) # -1 should be igonored (filler words)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,10692,-1_species_google_scholar_pubmed,"[species, google, scholar, pubmed, cell, scopu...",[meanings vegetation condition david keith emm...
1,0,362,0_ref_vol_cross_parameter,"[ref, vol, cross, parameter, parameters, stoch...",[next article description random field means c...
2,1,318,1_fish_fisheries_fishing_spawning,"[fish, fisheries, fishing, spawning, fishery, ...",[anthropogenic climate change resulted warming...
3,2,280,2_soil_soils_microbial_biochar,"[soil, soils, microbial, biochar, emissions, s...",[results reveal different responses soil multi...
4,3,260,3_students_education_learning_teachers,"[students, education, learning, teachers, stud...",[literature post pandemic education suggests b...
5,4,251,4_lung_ipf_pulmonary_alveolar,"[lung, ipf, pulmonary, alveolar, airway, copd,...",[abstract diseases idiopathic pulmonary fibros...
6,5,230,5_auxin_arabidopsis_aba_plants,"[auxin, arabidopsis, aba, plants, ethylene, st...",[drought stress responses vascular plants comp...
7,6,225,6_home_movement_habitat_wildlife,"[home, movement, habitat, wildlife, wolf, prey...",[pattern social spacing small mammals differs ...
8,7,216,7_protein_bioh_proteins_structural,"[protein, bioh, proteins, structural, structur...",[predicting function unknown protein essential...
9,8,204,8_conservation_restoration_biodiversity_ecosystem,"[conservation, restoration, biodiversity, ecos...",[ten years ago pages conservation biology whit...


In [47]:
len(freq)

317

In [48]:
topic_model.get_topic(0)  # Select the most frequent topic

[('fisheries', np.float64(0.022813323275379965)),
 ('fish', np.float64(0.022342186792597113)),
 ('fishing', np.float64(0.01239821167432394)),
 ('fishery', np.float64(0.010582425545975008)),
 ('spawning', np.float64(0.01034868327013018)),
 ('stock', np.float64(0.009663524025170412)),
 ('fishes', np.float64(0.009625794871663266)),
 ('management', np.float64(0.008752881117356044)),
 ('marine', np.float64(0.008698898155550826)),
 ('reef', np.float64(0.00858907839962417))]

## Visualize Topics

NOTE CONSIDER: (https://github.com/cpsievert/LDAvis):

In [ ]:
fig = topic_model.visualize_topics()
fig.write_html(
    "topics_visualization.html",
    config={'responsive': True, 'displayModeBar': True}
)

In [ ]:
fig_2 =topic_model.visualize_hierarchy(top_n_topics=100)
fig_2.write_html("topics_hierarchy.html")

## Visualize Terms

In [52]:
fig_2 =topic_model.visualize_barchart(top_n_topics=10)
fig_2.write_html("topics_barchart.html")



In [53]:

fig_2 = topic_model.visualize_term_rank()
fig_2.write_html("topics_term_rank.html")


## Update Topics

In [8]:
df2["topics"] = topics

In [10]:
ls

 Volume in drive C is Windows
 Volume Serial Number is DA97-C00E

 Directory of c:\Users\shaur\OneDrive\Desktop\Research Gap Identification

06/03/2026  05:43 PM    <DIR>          .
06/03/2026  10:33 PM    <DIR>          ..
06/02/2026  11:10 PM                36 .gitignore
05/24/2026  06:21 PM    <DIR>          .venv
06/02/2026  10:24 PM       412,952,642 abstract_final.csv
06/02/2026  10:32 PM       414,547,948 articles_clean_text_eng.csv
06/02/2026  10:35 PM       124,538,777 articles_with_sentences and words1.csv
06/03/2026  10:35 PM           462,565 CORD_Bib_Analysis Final.ipynb
06/02/2026  11:32 PM    <DIR>          Graphs
05/24/2026  06:20 PM     1,648,942,196 metadata.csv
06/02/2026  11:32 PM             1,623 README.md
06/02/2026  10:48 AM       404,001,731 scraped.csv
               8 File(s)  3,005,447,518 bytes
               4 Dir(s)  21,254,778,880 bytes free


In [11]:
df2.to_csv("articles_with_topics.csv",index=None)

# Analysis 

In [12]:
import pandas as pd
df2 = pd.read_csv("articles_with_topics.csv")

In [13]:
df2.columns

Index(['openalex_id', 'title', 'doi', 'abstract', 'publish_time', 'authors',
       'journal', 'doi.1', 'pmcid', 'pubmed_id', 'publish_time_new', 'lang',
       'clean_text', 'sentences', 'words', 'topics'],
      dtype='object')

In [14]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def ubt(x):
    word_data = x
    #word_data = "lime soda is the best selling item in fast food stores"

    # load nltk's stop word list
    stop_words = list(stopwords.words('english'))
    # extend the stop words list
    #stop_words.extend(["best", "selling", "item", "fast"])

    # tokenise the string and remove stop words
    word_tokens = word_tokenize(word_data)
    clean_word_data = [w for w in word_tokens if not w.lower() in stop_words]

    # get bigrams
    bigrams_list = [" ".join(item) for item in nltk.bigrams(clean_word_data)]
    #print(bigrams_list)

    # get trigrams
    trigrams_list = [" ".join(item) for item in nltk.trigrams(clean_word_data)]
    #print(trigrams_list)
    lst = []
    lst.extend(clean_word_data)
    lst.extend(bigrams_list)
    lst.extend(trigrams_list)
    from collections import Counter
    cnt = Counter(lst)
    del cnt["."]
    return cnt.most_common(50)

In [15]:
import re
dst =""
for i in range(191):
    st = " ".join(df2[df2.topics==i]["sentences"].tolist())
    st = re.sub(r'[^a-zA-Z0-9. ]', '', st)
    dst+="topic-"+str(i)
    dst +="\n"
    dst += str(ubt(st))
    dst +="\n"

print(dst)

topic-0
[('..', 918), ('model', 287), ('uncertainty', 239), ('models', 231), ('parameters', 218), ('systems', 210), ('parameter', 199), ('unknown', 198), ('data', 181), ('system', 126), ('methods', 110), ('experimental', 110), ('analysis', 103), ('method', 101), ('biological', 97), ('biology', 91), ('approach', 91), ('using', 91), ('problem', 87), ('estimation', 87), ('bayesian', 86), ('design', 80), ('equations', 79), ('uncertainties', 77), ('used', 72), ('space', 66), ('nonlinear', 63), ('dynamics', 61), ('stochastic', 61), ('differential', 60), ('many', 59), ('mathematical', 58), ('networks', 58), ('uncertain', 57), ('often', 55), ('unknown parameters', 55), ('results', 54), ('systems biology', 54), ('computational', 52), ('values', 51), ('network', 51), ('problems', 51), ('control', 51), ('based', 50), ('state', 49), ('time', 48), ('modeling', 47), ('framework', 47), ('experimental data', 47), ('statistical', 45)]
topic-1
[('..', 540), ('species', 157), ('fish', 152), ('fisheries',

In [16]:
with open('Topics Keywords with frequencies.txt', 'w') as f:
    f.write(dst)

# After Manually Labelled

In [17]:
import pandas as pd
df = pd.read_csv("articles_with_topics.csv")
df.head(3)

,openalex_id,title,doi,abstract,publish_time,authors,journal,doi.1,pmcid,pubmed_id,publish_time_new,lang,clean_text,sentences,words,topics
0,https://openalex.org/W2160719116,Synthetic biology for the directed evolution o...,https://doi.org/10.1039/c4cs00351a,the amino acid sequence of a protein affects b...,2014-12-15,Andrew Currin; Neil Swainston; Philip J. Day; ...,Chemical Society Reviews,https://doi.org/10.1039/c4cs00351a,NaN,https://pubmed.ncbi.nlm.nih.gov/25503938,2014-12-15,en,amino acid sequence protein affects structure ...,enzymologists distinguish binding (kd) and cat...,unknown,7
1,https://openalex.org/W3016143870,Space Radiation Biology for âLiving in Spaceâ,https://doi.org/10.1155/2020/4703286,space travel has advanced significantly over t...,2020-01-01,Satoshi Furukawa; Aiko Nagamatsu; Mitsuru Neno...,BioMed Research International,https://doi.org/10.1155/2020/4703286,NaN,https://pubmed.ncbi.nlm.nih.gov/32337251,2020-01-01,en,space travel advanced significantly last six d...,future research that furthers our understandin...,future research,36
2,https://openalex.org/W4366735078,Perspectives for plant biology in space and an...,https://doi.org/10.1038/s41526-023-00315-x,advancements in plant space biology are requir...,2023-08-21,Veronica De Micco; Giovanna Aronne; Nicol Capl...,npj Microgravity,https://doi.org/10.1038/s41526-023-00315-x,NaN,https://pubmed.ncbi.nlm.nih.gov/37604914,2023-08-21,en,advancements plant space biology required real...,this apparent paradox is a current research ch...,knowledge gap,207


In [18]:
len(df)

25520

In [20]:
import ast
import re

generic_terms = {
    
    "study", "result", "results", "analysis",
    "review", "data", "effect", "effects", "case", "cases", "using",
    "new", "also", "may", "could", "among", "based"
}

def make_topic_label(word_counts, max_words=2):
    terms = []
    for word, _ in word_counts:
        word = re.sub(r"[^a-zA-Z0-9 ]", "", str(word)).strip().lower()
        if not word:
            continue
        if word in generic_terms:
            continue
        if len(word) <= 2:
            continue
        terms.append(word)

    if not terms:
        return "General"

    label_terms = terms[:max_words]
    return " ".join(term.title() for term in label_terms)

tp = {-1: "Unlabelled"}

current_topic = None
for line in dst.splitlines():
    line = line.strip()
    if line.startswith("topic-"):
        current_topic = int(line.split("-", 1)[1])
        continue

    if current_topic is not None and line.startswith("["):
        try:
            word_counts = ast.literal_eval(line)
            tp[current_topic] = make_topic_label(word_counts)
        except Exception:
            tp[current_topic] = "General"
        current_topic = None

print(tp)

{-1: 'Unlabelled', 0: 'Model Uncertainty', 1: 'Species Fish', 2: 'Soil Microbial', 3: 'Research Learning', 4: 'Lung Pulmonary', 5: 'Cell Plant', 6: 'Species Research', 7: 'Protein Proteins', 8: 'Research Conservation', 9: 'Mitochondrial Mitochondria', 10: 'Unknown Selection', 11: 'Sperm Cells', 12: 'Coral Corals', 13: 'Chromatin Unclear', 14: 'Cardiac Studies', 15: 'Fungi Fungal', 16: 'Retinal Cells', 17: 'Bee Species', 18: 'Bone Cells', 19: 'Research Future', 20: 'Cell Cells', 21: 'Cancer Mirnas', 22: 'Gut Microbiome', 23: 'Water Uncertainty', 24: 'Whales Species', 25: 'Species Research', 26: 'Studies Placental', 27: 'Gut Intestinal', 28: 'Cancer Breast', 29: 'Cartilage Hcgp39', 30: 'Forest Carbon', 31: 'Model Models', 32: 'Liver Hepatic', 33: 'Species Unknown', 34: 'Species Uncertainty', 35: 'Research Science', 36: 'Radiation Space', 37: 'Tau Disease', 38: 'Plant Pathogen', 39: 'Space Microgravity', 40: 'Research Biology', 41: 'Uncertainty Phylogenetic', 42: 'Virus Viral', 43: 'Genes

In [23]:
df["Topic Labels"] = df["topics"].fillna(-1).astype(int).map(lambda x: tp.get(x, "General"))

In [ ]:
df.groupby(by=["Topic Labels"])["title"].count()

Topic Labels
000 Per               40
Aging Studies         31
Amphibian Species     27
Amyloid Disease       32
Asd Autism            41
                    ... 
Visual Whether        52
Vocal Species         34
Water Uncertainty    114
Whales Species       113
Wound Healing         37
Name: title, Length: 180, dtype: int64